In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('emotions.txt', sep=';', header=None, names=['text', 'emotions'])
df.head()

,text,emotions
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness


In [3]:
df.isnull().sum()

text        0
emotions    0
dtype: int64

In [4]:
unique_emotions = df['emotions'].unique()
emotion_number = {}
i = 0

for emo in unique_emotions:
    emotion_number[emo] = i
    i += 1

df['emotions'] = df['emotions'].map(emotion_number)
df

,text,emotions
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0
...,...,...
1995,i just keep feeling like someone is being unki...,3
1996,im feeling a little cranky negative after this...,3
1997,i feel that i am useful to my people and that ...,1
1998,im feeling more comfortable with derby i feel ...,1


In [5]:
# Step 1 : Convert to Lower Case
df['text'] = df['text'].apply(lambda x : x.lower())
df.head()

,text,emotions
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


In [6]:
# Step 2 : Remove Punctuations
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [7]:
df['text'] = df['text'].apply(remove_punc)
df.head()

,text,emotions
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


In [8]:
# Step 3 : Remove Numbers from Text
def remove_numbers(txt):
    new = ""
    for char in txt:
        if not char.isdigit():
            new += char
    return new

df['text'] = df['text'].apply(remove_numbers)

In [9]:
# Step 4 : Remove URLs
# Step 5 : Remove HTML Tags (Helpful in  Web Scrapped Data)

# Step 6: Remove Emoji's
def remove_emojis(txt):
    new = ""
    for char in txt:
        if char.isascii():
          new += char
    return new

df['text'] = df['text'].apply(remove_emojis)

In [10]:
# Step - 7 : Remove Stop Words (the, is, was etc...)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [11]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Soni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Soni\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [13]:
df.loc[1]['text']

'im updating my blog because i feel shitty'

In [14]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []

    for i in words:
        if not i in stop_words: 
            cleaned.append(i)
    
    return ' '.join(cleaned)

In [15]:
df['text'] = df['text'].apply(remove)
df.head()

,text,emotions
0,im feeling rather rotten im ambitious right,0
1,im updating blog feel shitty,0
2,never make separate ever want feel like ashamed,0
3,left bouquet red yellow tulips arm feeling sli...,1
4,feeling little vain one,0


In [16]:
df.loc[1]['text']

'im updating blog feel shitty'

In [17]:
#Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

In [18]:
documents = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is great"
]

In [19]:
vectorizer = CountVectorizer(ngram_range=(2,2))

In [20]:
X = vectorizer.fit_transform(documents)

print("Vocabulary : ", vectorizer.get_feature_names_out())
print("\n Bag of Word Matrix : \n", X.toarray())

Vocabulary :  ['is great' 'is the' 'love pasta' 'love pizza' 'pasta is' 'pizza is'
 'the best']

 Bag of Word Matrix : 
 [[0 0 0 1 0 0 0]
 [0 1 0 0 0 1 1]
 [0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0]]


In [21]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tf_vectorizer = TfidfVectorizer()

In [22]:
Y = tf_vectorizer.fit_transform(documents)

print("Vocabulary : ", tf_vectorizer.get_feature_names_out())
print("\nTF-IDF Matrix : \n", Y.toarray())

Vocabulary :  ['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']

TF-IDF Matrix : 
 [[0.         0.         0.         0.70710678 0.         0.70710678
  0.        ]
 [0.55528266 0.         0.43779123 0.         0.         0.43779123
  0.55528266]
 [0.         0.         0.         0.70710678 0.70710678 0.
  0.        ]
 [0.         0.66767854 0.52640543 0.         0.52640543 0.
  0.        ]]


In [23]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['emotions']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [25]:
bow_vectorizer = CountVectorizer()

In [26]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [27]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

model = MultinomialNB(alpha=0.3)

In [28]:
model.fit(X_train_bow, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.3
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [29]:
y_pred = model.predict(X_test_bow)

print("Accuracy using Bag of Words : " , accuracy_score(y_test, y_pred))

Accuracy using Bag of Words :  0.66


In [30]:
#Using TF-IDF Now Finding Accuracy
tfidf_vectorizer = TfidfVectorizer()

In [31]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [32]:
nb2_model = MultinomialNB(alpha=0.1)

In [33]:
nb2_model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.1
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [34]:
y_pred_nb2 = nb2_model.predict(X_test_tfidf)

print("Accuracy : ", accuracy_score(y_pred_nb2, y_test))

Accuracy :  0.6825


In [48]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()

In [49]:
lr_model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [50]:
y_pred_lr = lr_model.predict(X_test_tfidf)

print("Accuracy : ", accuracy_score(y_pred_lr, y_test))

Accuracy :  0.6375
